# FurEverSafe — LoRA → GGUF Conversion
This notebook:
1. Installs dependencies
2. Loads TinyLlama base + your LoRA adapter
3. Merges them into a single model
4. Converts to GGUF (f16, then optionally quantizes to Q4_K_M)
5. Outputs the `.gguf` file ready for download

**Upload your `fureversafe_lora_model/` folder as a Kaggle Dataset before running.**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers peft accelerate torch sentencepiece
!pip install -q 'peft>=0.19.1'

In [ ]:
# ── Cell 2: Clone llama.cpp (for convert_hf_to_gguf.py) ──────────────────────
import os

if not os.path.exists('/kaggle/working/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
    !pip install -q -r /kaggle/working/llama.cpp/requirements.txt
    print('llama.cpp cloned ✅')
else:
    print('llama.cpp already present ✅')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
import os

BASE_MODEL      = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Path to the uploaded Kaggle dataset containing your LoRA adapter files.
# After uploading your dataset, Kaggle mounts it at /kaggle/input/<dataset-name>/
# Update LORA_PATH to match your dataset name:
LORA_PATH = '/kaggle/input/fureversafe-lora'

MERGED_PATH     = '/kaggle/working/merged_model'
GGUF_OUTPUT_F16 = '/kaggle/working/fureversafe_f16.gguf'
GGUF_OUTPUT_Q4  = '/kaggle/working/fureversafe_q4_k_m.gguf'
CONVERT_SCRIPT  = '/kaggle/working/llama.cpp/convert_hf_to_gguf.py'
QUANTIZE_BIN    = '/kaggle/working/llama.cpp/build/bin/llama-quantize'

print('Config set ✅')
print(f'  LoRA path exists: {os.path.exists(LORA_PATH)}')
print(f'  Adapter files: {os.listdir(LORA_PATH) if os.path.exists(LORA_PATH) else "NOT FOUND"}')

In [ ]:
# ── Cell 4: Merge LoRA into base model ───────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map='cpu'
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print('Base model loaded ✅')

print('Attaching LoRA adapter...')
model = PeftModel.from_pretrained(base_model, LORA_PATH)
print('LoRA attached ✅')

print('Merging weights (this takes ~1-2 min)...')
model = model.merge_and_unload()
print('Merge complete ✅')

print(f'Saving merged model to {MERGED_PATH}...')
os.makedirs(MERGED_PATH, exist_ok=True)
model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)
print('Saved ✅')

In [ ]:
# ── Cell 5: Convert merged model to GGUF (f16) ───────────────────────────────
import subprocess, sys

print('Converting to GGUF f16...')
result = subprocess.run([
    sys.executable, CONVERT_SCRIPT,
    MERGED_PATH,
    '--outfile', GGUF_OUTPUT_F16,
    '--outtype', 'f16'
], capture_output=False)

if result.returncode != 0:
    raise RuntimeError('GGUF conversion failed!')

size_mb = os.path.getsize(GGUF_OUTPUT_F16) / 1024**2
print(f'GGUF f16 created ✅  ({size_mb:.1f} MB) → {GGUF_OUTPUT_F16}')

In [ ]:
# ── Cell 6: (Optional) Quantize to Q4_K_M for smaller size ──────────────────
# Q4_K_M = ~700 MB vs ~2.2 GB for f16, with minimal quality loss
# Skip this cell if you prefer f16.

print('Building llama-quantize tool...')
os.makedirs('/kaggle/working/llama.cpp/build', exist_ok=True)
!cd /kaggle/working/llama.cpp/build && cmake .. -DCMAKE_BUILD_TYPE=Release -DLLAMA_NATIVE=OFF 2>&1 | tail -5
!cd /kaggle/working/llama.cpp/build && make llama-quantize -j$(nproc) 2>&1 | tail -5
print('Build done ✅')

print('Quantizing to Q4_K_M...')
result = subprocess.run([
    QUANTIZE_BIN,
    GGUF_OUTPUT_F16,
    GGUF_OUTPUT_Q4,
    'Q4_K_M'
], capture_output=False)

if result.returncode != 0:
    print('Quantization failed — f16 file is still usable.')
else:
    size_mb = os.path.getsize(GGUF_OUTPUT_Q4) / 1024**2
    print(f'Q4_K_M created ✅  ({size_mb:.1f} MB) → {GGUF_OUTPUT_Q4}')

In [ ]:
# ── Cell 7: Summary & Download Links ─────────────────────────────────────────
print('=== Output Files ===')
for path in [GGUF_OUTPUT_F16, GGUF_OUTPUT_Q4]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f'  ✅ {os.path.basename(path)}  ({size_mb:.1f} MB)')
    else:
        print(f'  ✗  {os.path.basename(path)} not found')

print()
print('Download the .gguf file(s) from the Kaggle Output tab (right panel).')
print('Recommended: use Q4_K_M for production (~700 MB, fast CPU inference).')